# Matrix Factorization with PyTorch

In [1]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# Check if GPU is available and set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


## Read the data and print descriptive statistics

In [3]:
df = pd.read_csv('data/ml-latest-small/ratings.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [5]:
df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [6]:
df.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


## Data Preprocessing

#### Prepare the user and item indices

In [7]:
df['mId'] = pd.factorize(df['movieId'])[0]

In [8]:
df['uId'] = df['userId'].values - 1

In [9]:
df.head()

,userId,movieId,rating,timestamp,mId,uId
0,1,1,4.0,964982703,0,0
1,1,3,4.0,964981247,1,0
2,1,6,4.0,964982224,2,0
3,1,47,5.0,964983815,3,0
4,1,50,5.0,964982931,4,0


In [10]:
user_indices = torch.tensor(df['uId'].values, dtype=torch.long).to(device)
item_indices = torch.tensor(df['mId'].values, dtype=torch.long).to(device)
ratings = torch.tensor(df['rating'].values, dtype=torch.float32).to(device)
print(len(user_indices), len(item_indices), len(ratings))

100836 100836 100836


In [11]:
# Dimensions
num_users = df['uId'].max() + 1
num_items = df['mId'].max() + 1
num_features = 128  # Number of latent features
print(num_users, num_items, num_features)

610 9724 128


### Train/test split

In [12]:
# Convert tensors to numpy arrays for scikit-learn
user_indices_np = user_indices.cpu().numpy()
item_indices_np = item_indices.cpu().numpy()
ratings_np = ratings.cpu().numpy()

# Split the data
train_user_indices, test_user_indices, train_item_indices, test_item_indices, train_ratings, test_ratings = train_test_split(
    user_indices_np, item_indices_np, ratings_np, test_size=0.2, random_state=42
)

# Convert back to tensors
train_user_indices = torch.tensor(train_user_indices).to(device)
train_item_indices = torch.tensor(train_item_indices).to(device)
train_ratings = torch.tensor(train_ratings).to(device)

test_user_indices = torch.tensor(test_user_indices).to(device)
test_item_indices = torch.tensor(test_item_indices).to(device)
test_ratings = torch.tensor(test_ratings).to(device)

### Create batch

In [13]:
def batch(user_indices, item_indices, ratings, batch_size):
    ix = torch.randint(0, len(user_indices), (batch_size, ))
    ui = user_indices[ix]
    ii = item_indices[ix]
    r = ratings[ix]
    return ui, ii, r

## Model training on the training dataset

In [40]:
def train(model):
    # Loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.0001)

    # Training loop
    num_epochs = 50
    num_batches = len(train_user_indices) // 256
    batch_size = 128
    for epoch in range(num_epochs):
        model.train()
    
        losses = []
        for _ in range(num_batches):
            # Get a random batch of data
            batch_user_indices, batch_item_indices, batch_ratings = batch(train_user_indices, train_item_indices, train_ratings, batch_size)

            # Forward pass
            predicted_ratings = model(batch_user_indices, batch_item_indices)

            # Compute the loss
            loss = criterion(predicted_ratings, batch_ratings)
            losses.append(loss.item())

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {torch.sqrt(torch.tensor(losses).mean()):.4f}')

## Model evaluation on the test dataset

In [15]:
def eval(model):
    # Evaluate the final reconstruction
    criterion = nn.MSELoss()

    model.eval()
    with torch.no_grad():
        predicted_ratings = model(test_user_indices, test_item_indices)
        test_loss = criterion(predicted_ratings, test_ratings)
        print(f'Test RMSE Loss: {torch.sqrt(test_loss):.4f}')

## Train and evaluate

In [16]:
def train_eval(model_class):
    model = model_class(num_users, num_items, num_features)
    model.to(device)

    train(model)
    eval(model)

## Vanilla MF Model

In [17]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, num_features):
        super(MatrixFactorization, self).__init__()
        self.user_embeddings = nn.Embedding(num_users, num_features)
        self.item_embeddings = nn.Embedding(num_items, num_features)
        self.user_embeddings.weight.data.uniform_(0., 0.05)
        self.item_embeddings.weight.data.uniform_(0., 0.05)

    def forward(self, user_indices, item_indices):
        user_vecs = self.user_embeddings(user_indices)
        item_vecs = self.item_embeddings(item_indices)
        
        predicted_values = (user_vecs * item_vecs).sum(dim=1)
        return predicted_values

## MF with Global Biases

In [18]:
class MatrixFactorizationGB(nn.Module):
    def __init__(self, num_users, num_items, num_features):
        super(MatrixFactorizationGB, self).__init__()
        self.user_embeddings = nn.Embedding(num_users, num_features)
        self.item_embeddings = nn.Embedding(num_items, num_features)
        self.user_embeddings.weight.data.uniform_(0., 0.05)
        self.item_embeddings.weight.data.uniform_(0., 0.05)

        self.user_bias = nn.Parameter(torch.zeros(num_users))
        self.item_bias = nn.Parameter(torch.zeros(num_items))
        self.offset = nn.Parameter(torch.zeros(1))

    def forward(self, user_indices, item_indices):
        user_vecs = self.user_embeddings(user_indices)
        item_vecs = self.item_embeddings(item_indices)
        
        predicted_values = (user_vecs * item_vecs).sum(dim=1) + self.user_bias[user_indices] + self.item_bias[item_indices] + self.offset
        return predicted_values

## MF with Neural Network

In [50]:
class MatrixFactorizationNN(nn.Module):
    def __init__(self, num_users, num_items, num_features):
        super(MatrixFactorizationNN, self).__init__()
        
        self.user_embeddings = nn.Embedding(num_users, num_features)
        self.item_embeddings = nn.Embedding(num_items, num_features)

        self.user_mlp = nn.Embedding(num_users, num_features)
        self.item_mlp = nn.Embedding(num_items, num_features)
        self.hidden_dims = (256, 64, 16)
        self.dropout = 0.1

        self.user_bias = nn.Parameter(torch.zeros(num_users))
        self.item_bias = nn.Parameter(torch.zeros(num_items))
        self.offset = nn.Parameter(torch.zeros(1))

        layers = []
        input_dim = num_features * 2

        for hidden_dim in self.hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.Tanh())
            layers.append(nn.Dropout(self.dropout))
            input_dim = hidden_dim

        self.mlp = nn.Sequential(*layers)

        # Final layer combines MF dot-product + MLP representation
        self.output = nn.Linear(self.hidden_dims[-1] + 1, 1)

        self.apply(self._init_weights)


    def _init_weights(self, module):
        if isinstance(module, nn.Linear):            
            torch.nn.init.xavier_uniform_(module.weight, gain=nn.init.calculate_gain('tanh'))
            torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.uniform_(module.weight, 0., 0.05)


    def forward(self, user_indices, item_indices):
        user_vecs = self.user_embeddings(user_indices)
        item_vecs = self.item_embeddings(item_indices)
        mf_out = (user_vecs * item_vecs).sum(dim=1, keepdim=True)

        user_mlp = self.user_mlp(user_indices)
        item_mlp = self.item_mlp(item_indices)
        x = torch.cat([user_mlp, item_mlp], dim=1)
        mlp_out = self.mlp(x)

        combined = self.output(torch.cat([mf_out, mlp_out], dim=1))
        out = combined + self.user_bias[user_indices].view(-1, 1) + self.item_bias[item_indices].view(-1, 1) + self.offset
        return out.squeeze(1)

## Compare models

In [51]:
for model_class in [MatrixFactorization, MatrixFactorizationGB, MatrixFactorizationNN]:
    print('*' * 50)
    print(f'Training and evaluating {model_class.__name__}...')
    train_eval(model_class)

**************************************************
Training and evaluating MatrixFactorization...
Epoch [10/50], Loss: 1.0869
Epoch [20/50], Loss: 0.8640
Epoch [30/50], Loss: 0.7782
Epoch [40/50], Loss: 0.7522
Epoch [50/50], Loss: 0.7341
Test RMSE Loss: 1.2450
**************************************************
Training and evaluating MatrixFactorizationGB...
Epoch [10/50], Loss: 0.6516
Epoch [20/50], Loss: 0.6195
Epoch [30/50], Loss: 0.6014
Epoch [40/50], Loss: 0.5878
Epoch [50/50], Loss: 0.5859
Test RMSE Loss: 0.9010
**************************************************
Training and evaluating MatrixFactorizationNN...
Epoch [10/50], Loss: 0.6727
Epoch [20/50], Loss: 0.6226
Epoch [30/50], Loss: 0.6015
Epoch [40/50], Loss: 0.5954
Epoch [50/50], Loss: 0.5936
Test RMSE Loss: 0.8994
